# PARTIE 2 : TRAITEMENT BATCH AVEC SPARKCORE (RDD)

## 1. Objectif du Notebook
Dans cette partie, nous manipulons l'API bas niveau de Spark (**RDD : Resilient Distributed Datasets**) pour analyser un historique de 50 000 commandes stocké dans MinIO (`s3a://lake/bronze/history/orders_history.jsonl`).

### Notions clés abordées :
- **Transformations paresseuses (*lazy*)** : `map`, `filter`, `reduceByKey`, `sortBy`.
- **Actions qui déclenchent le calcul** : `collect`, `take`, `count`.
- **Lignée logique (*Lineage / DAG*)** : chaîne de dépendances permettant la tolérance aux pannes sans duplication de données.
- **Shuffle distribué** : réorganisation des partitions au travers du réseau provoquée par `reduceByKey`.

## 2. Initialisation de la SparkSession et du SparkContext

In [1]:
import os
import sys
import json
import socket
from pyspark.sql import SparkSession

# 1. Configuration de l'environnement Python et Java (macOS ARM64 Apple Silicon)
os.environ["JAVA_HOME"] = "/opt/homebrew/opt/openjdk@17"
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

# 2. Détection dynamique de l'IP hôte pour la communication bidirectionnelle Driver <-> Workers Docker & MinIO S3
s = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
try:
    s.connect(("8.8.8.8", 80))
    host_ip = s.getsockname()[0]
except Exception:
    host_ip = "127.0.0.1"
finally:
    s.close()

minio_endpoint = f"http://{host_ip}:9000"

# 3. Initialisation de la SparkSession connectée au véritable cluster Spark distribué (Master Standalone)
spark = (
    SparkSession.builder
    .appName("Partie2_Batch_SparkCore_RDD")
    .master("spark://localhost:7077")
    .config("spark.driver.host", "host.docker.internal")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.driver.port", "7078")
    .config("spark.blockManager.port", "7079")
    .config("spark.jars.packages", "org.apache.hadoop:hadoop-aws:3.3.4,com.amazonaws:aws-java-sdk-bundle:1.12.262")
    .config("spark.hadoop.fs.s3a.endpoint", minio_endpoint)
    .config("spark.hadoop.fs.s3a.access.key", "admin")
    .config("spark.hadoop.fs.s3a.secret.key", "password123")
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .getOrCreate()
)

sc = spark.sparkContext
sc.setLogLevel("WARN")
print("[OK] SparkSession RDD initialisée avec succès.")
print(f" • Version Spark        : {spark.version}")
print(f" • Master connecté      : {sc.master}")
print(f" • Nom de l'application : {sc.appName}")
print(f" • Cœurs distribués     : {sc.defaultParallelism} cœurs (sur les 2 Workers Docker)")
print(f" • Endpoint MinIO S3A   : {minio_endpoint}")


[OK] SparkSession RDD initialisée avec succès.
 • Version Spark        : 3.5.0
 • Master connecté      : spark://localhost:7077
 • Nom de l'application : Partie2_Batch_SparkCore_RDD
 • Cœurs distribués     : 2 cœurs (sur les 2 Workers Docker)
 • Endpoint MinIO S3A   : http://192.168.1.18:9000


## 3. Chargement de l'Historique depuis MinIO et Filtrage
Nous chargeons le fichier JSON Lines depuis la couche Bronze du Data Lake avec `sc.textFile()` et nous filtrons uniquement les commandes dont le statut est `DELIVERED`.

In [2]:
# Transformation 1 : Lecture du fichier brut depuis MinIO (retourne un RDD de chaînes)
rdd_brut = sc.textFile("s3a://lake/bronze/history/orders_history.jsonl")

# Transformation 2 : Parsing JSON de chaque ligne en dictionnaire Python
rdd_json = rdd_brut.map(lambda line: json.loads(line))

# Transformation 3 : Filtrage des commandes avec le statut 'DELIVERED'
rdd_delivered = rdd_json.filter(lambda x: x.get("status") == "DELIVERED")

# Mise en cache mémoire car ce RDD sera réutilisé dans plusieurs calculs successifs
rdd_delivered.cache()

total_livrees = rdd_delivered.count()  # Action : déclenche l'exécution et le cache
print(f"[OK] Nombre total de commandes livrées (DELIVERED) : {total_livrees:,}")

[OK] Nombre total de commandes livrées (DELIVERED) : 30,613


## 4. Chiffre d'Affaires Total par Ville (`map` + `reduceByKey`)
Pour chaque commande livrée, nous émettons un couple `(ville, montant)` via `map()`, puis nous agrégeons les montants par ville avec `reduceByKey()`.

In [3]:
# Couple clé/valeur : (clé=ville, valeur=amount_mad)
ca_par_ville = (
    rdd_delivered
    .map(lambda x: (x["city"], x["amount_mad"]))
    .reduceByKey(lambda a, b: a + b)
)

print("--- Chiffre d'Affaires Total par Ville ---")
for ville, ca in ca_par_ville.collect():
    print(f" • {ville:<12} : {ca:>12,.2f} MAD")

--- Chiffre d'Affaires Total par Ville ---
 • Tanger       : 1,403,290.44 MAD
 • Rabat        : 1,681,438.04 MAD
 • Casablanca   : 1,719,956.11 MAD
 • Marrakech    : 1,992,914.28 MAD
 • Agadir       : 1,319,097.74 MAD


## 5. Panier Moyen par Ville en UN SEUL PASSAGE
**Exigence du professeur :** Calculer la somme et le nombre de commandes en un seul passage pour éviter de reparcourir deux fois les données.  
Nous émettons un tuple `(ville, (montant, 1))` : `reduceByKey` additionne simultanément les montants et les compteurs.

In [4]:
# Clé = ville, Valeur = (montant, 1)
panier_stats = (
    rdd_delivered
    .map(lambda x: (x["city"], (x["amount_mad"], 1)))
    .reduceByKey(lambda a, b: (a[0] + b[0], a[1] + b[1]))
)

# Calcul de la moyenne : somme / effectif
panier_moyen = panier_stats.mapValues(lambda v: (v[0] / v[1], v[0], v[1]))

print("--- Panier Moyen et Volume par Ville (En 1 seul passage) ---")
for ville, (moyenne, ca, nb) in panier_moyen.collect():
    print(f" • {ville:<12} : Panier Moyen = {moyenne:>7.2f} MAD | Commandes = {nb:>5} | Total = {ca:>11,.2f} MAD")

--- Panier Moyen et Volume par Ville (En 1 seul passage) ---
 • Tanger       : Panier Moyen =  265.32 MAD | Commandes =  5289 | Total = 1,403,290.44 MAD
 • Rabat        : Panier Moyen =  263.38 MAD | Commandes =  6384 | Total = 1,681,438.04 MAD
 • Casablanca   : Panier Moyen =  267.24 MAD | Commandes =  6436 | Total = 1,719,956.11 MAD
 • Marrakech    : Panier Moyen =  264.56 MAD | Commandes =  7533 | Total = 1,992,914.28 MAD
 • Agadir       : Panier Moyen =  265.36 MAD | Commandes =  4971 | Total = 1,319,097.74 MAD


## 6. Top 5 des Restaurants par Nombre de Commandes Livrées

In [5]:
# Comptage par restaurant_id puis tri décroissant
top5_restaurants = (
    rdd_delivered
    .map(lambda x: (x["restaurant_id"], 1))
    .reduceByKey(lambda a, b: a + b)
    .sortBy(lambda pair: pair[1], ascending=False)
    .take(5)
)

print("--- TOP 5 des Restaurants les plus sollicités ---")
for rank, (rest_id, count) in enumerate(top5_restaurants, 1):
    print(f" #{rank} : Restaurant {rest_id} avec {count} commandes livrées")

--- TOP 5 des Restaurants les plus sollicités ---
 #1 : Restaurant R-123 avec 197 commandes livrées
 #2 : Restaurant R-196 avec 181 commandes livrées
 #3 : Restaurant R-056 avec 180 commandes livrées
 #4 : Restaurant R-084 avec 180 commandes livrées
 #5 : Restaurant R-182 avec 179 commandes livrées


## 7. Lineage Graph (*toDebugString*) et Explications Pédagogiques

In [6]:
# RDD final du Top 5 avant l'action take(5)
rdd_top = (
    rdd_delivered
    .map(lambda x: (x["restaurant_id"], 1))
    .reduceByKey(lambda a, b: a + b)
    .sortBy(lambda pair: pair[1], ascending=False)
)

print("=== ARBRE DE LIGNÉE (LINEAGE GRAPH) ===")
debug_str = rdd_top.toDebugString()
print(debug_str.decode("utf-8") if isinstance(debug_str, bytes) else debug_str)


=== ARBRE DE LIGNÉE (LINEAGE GRAPH) ===
(2) PythonRDD[35] at RDD at PythonRDD.scala:53 []
 |  MapPartitionsRDD[34] at mapPartitions at PythonRDD.scala:160 []
 |  ShuffledRDD[33] at partitionBy at NativeMethodAccessorImpl.java:0 []
 +-(2) PairwiseRDD[32] at sortBy at <string>:6 []
    |  PythonRDD[31] at sortBy at <string>:6 []
    |  MapPartitionsRDD[28] at mapPartitions at PythonRDD.scala:160 []
    |  ShuffledRDD[27] at partitionBy at NativeMethodAccessorImpl.java:0 []
    +-(2) PairwiseRDD[26] at reduceByKey at <string>:5 []
       |  PythonRDD[25] at reduceByKey at <string>:5 []
       |  PythonRDD[2] at RDD at PythonRDD.scala:53 []
       |      CachedPartitions: 2; MemorySize: 1409.6 KiB; DiskSize: 0.0 B
       |  s3a://lake/bronze/history/orders_history.jsonl MapPartitionsRDD[1] at textFile at NativeMethodAccessorImpl.java:0 []
       |  s3a://lake/bronze/history/orders_history.jsonl HadoopRDD[0] at textFile at NativeMethodAccessorImpl.java:0 []


### Explications Pédagogiques des Notions Big Data :

#### 1. Transformations (*Lazy*) vs Actions
- **Transformations (`map`, `filter`, `reduceByKey`, `sortBy`)** : Elles sont **paresseuses (*lazy*)**. Spark ne calcule aucun résultat lorsqu'elles sont invoquées. Il enregistre simplement une étape dans le plan d'exécution (le DAG).
- **Actions (`count`, `collect`, `take`)** : Ce sont les seules opérations qui déclenchent réellement le calcul sur le cluster. Spark compile alors le DAG en stages physiques et soumet les tâches aux Executors.

#### 2. Rôle du Lineage dans la Tolérance aux Pannes
- Contrairement aux systèmes traditionnels qui répliquent les données sur disque pour résister aux pannes (comme HDFS avec un facteur 3), Spark stocke uniquement la **recette de reconstruction** (le *lineage*).
- Si un Worker tombe en panne et qu'une partition en mémoire est perdue, Spark ne recalcule pas tout le job : il réapplique simplement la chaîne de transformations depuis la source (MinIO S3) **uniquement pour la partition perdue**.

#### 3. Explication du Shuffle provoqué par `reduceByKey`
- Dans `reduceByKey(lambda a, b: a + b)`, Spark doit regrouper toutes les valeurs associées à une même clé (ex: la même ville ou le même restaurant), initialement dispersées sur différentes partitions.
- Cela nécessite un **Shuffle** : les Workers écrivent leurs données partielles sur disque local (Shuffle Write), puis s'échangent les blocs au travers du réseau (Shuffle Read) pour rassembler les mêmes clés sur le même nœud.
- `reduceByKey` est bien plus performant que `groupByKey` car il effectue un pré-regroupement local (*combiner*) sur chaque partition avant de transmettre les données sur le réseau, réduisant drastiquement le trafic réseau.